# Magneto-Coriolis Modes in a Closed Cylinder

This notebook provides a self-contained and reproducible derivation and numerical solution of magneto-Coriolis modes in a finite rotating cylinder. It is designed to be executed sequentially, with each stage of the mathematical formulation, discretization, eigensolution, validation, and post-processing documented explicitly.

The incompressible velocity and magnetic perturbations are represented using the classical toroidal-poloidal decomposition relative to the uniform axial direction $\mathbf e_z$. Their radial dependence is regularized analytically at the rotation axis and discretized using a Chebyshev spectral method. The resulting system is assembled as a generalized eigenvalue problem and solved with PETSc/SLEPc.

The numerical workflow is

$$
\text{MHD equations}
\longrightarrow
\text{toroidal-poloidal decomposition}
\longrightarrow
\text{axis regularization}
\longrightarrow
\text{Chebyshev discretization} \\
\longrightarrow
\text{generalized eigenvalue problem}
\longrightarrow
\text{SLEPc eigensolution}
\longrightarrow
\text{mode identification and validation}.
$$

For viscous calculations, the complete no-slip condition is imposed at the lateral wall, while the magnetic perturbation is matched to an exterior current-free field.

The computed modes are validated against the analytical Kelvin-Greenspan inertial-wave limit, ideal magneto-Coriolis frequencies, and the Alfvén-wave asymptote. Spectral convergence, boundary residuals, eigenvector overlap, and two-dimensional field reconstructions are used to assess the numerical solutions.

## 1. Physical problem and governing equations

We consider an incompressible, electrically conducting Newtonian fluid contained in a finite circular cylinder of radius $a$ and height $h$. The cylinder rotates uniformly at angular velocity

$$
\boldsymbol{\Omega}=\Omega\,\mathbf e_z,
$$

and is permeated by a uniform imposed axial magnetic field

$$
\mathbf B_0=B_0\,\mathbf e_z.
$$

The problem is studied in the reference frame rotating with the cylinder. Cylindrical coordinates are denoted by

$$
(s,\varphi,z),
\qquad
0\leq s\leq a,
\qquad
0\leq\varphi<2\pi,
\qquad
0\leq z\leq h.
$$

The velocity and magnetic fields are decomposed into a basic state and infinitesimal perturbations:

$$
\mathbf U=\mathbf v,
$$

$$
\mathbf B=\mathbf B_0+\mathbf b.
$$

The basic state is at rest in the rotating frame, and the imposed field is steady and spatially uniform. Neglecting terms quadratic in the perturbations, the dimensional linearized momentum and induction equations are

$$
\frac{\partial\mathbf v}{\partial t}
+
2\boldsymbol{\Omega}\times\mathbf v
=
-\frac{1}{\rho}\nabla p
+
\frac{1}{\rho\mu_0}
\left(\nabla\times\mathbf b\right)\times\mathbf B_0
+
\nu\nabla^2\mathbf v,
$$

$$
\frac{\partial\mathbf b}{\partial t}
=
\nabla\times
\left(
\mathbf v\times\mathbf B_0
\right)
+
\eta\nabla^2\mathbf b.
$$

Here, $\rho$ is the constant fluid density, $\mu_0$ is the magnetic permeability, $\nu$ is the kinematic viscosity, and $\eta$ is the magnetic diffusivity. The pressure $p$ includes the hydrodynamic pressure together with any potential centrifugal contribution absorbed into the basic state.

Both perturbation fields are solenoidal:

$$
\nabla\cdot\mathbf v=0,
$$

$$
\nabla\cdot\mathbf b=0.
$$

Because the imposed magnetic field is uniform and parallel to the rotation axis,

$$
\nabla\times
\left(
\mathbf v\times\mathbf B_0
\right)
=
B_0\frac{\partial\mathbf v}{\partial z},
$$

while the linearized Lorentz force may be written as

$$
\left(\nabla\times\mathbf b\right)\times\mathbf B_0
=
B_0\frac{\partial\mathbf b}{\partial z}
-
\nabla\left(B_0b_z\right).
$$

The gradient contribution can be absorbed into a modified pressure,

$$
\Pi
=
\frac{p}{\rho}
+
\frac{B_0b_z}{\rho\mu_0}.
$$

The linearized dimensional system therefore becomes

$$
\frac{\partial\mathbf v}{\partial t}
+
2\Omega\,\mathbf e_z\times\mathbf v
=
-\nabla\Pi
+
\frac{B_0}{\rho\mu_0}
\frac{\partial\mathbf b}{\partial z}
+
\nu\nabla^2\mathbf v,
$$

$$
\frac{\partial\mathbf b}{\partial t}
=
B_0\frac{\partial\mathbf v}{\partial z}
+
\eta\nabla^2\mathbf b,
$$

subject to

$$
\nabla\cdot\mathbf v
=
\nabla\cdot\mathbf b
=
0.
$$

---

### 1.1 Nondimensional formulation

Lengths are scaled by the cylinder radius $a$, time by the inverse rotation rate $\Omega^{-1}$, velocity by an arbitrary modal velocity scale $U$, and magnetic perturbations by

$$
B_0\frac{U}{\Omega a}.
$$

Thus,

$$
\mathbf x=a\,\mathbf x^*,
\qquad
t=\Omega^{-1}t^*,
\qquad
\mathbf v=U\,\mathbf v^*,
\qquad
\mathbf b=
B_0\frac{U}{\Omega a}\mathbf b^*.
$$

The modified pressure is scaled by $\rho\Omega aU$. After dropping the asterisks, the dimensionless equations are

$$
\frac{\partial\mathbf v}{\partial t}
+
2\mathbf e_z\times\mathbf v
=
-\nabla\Pi
+
Le^2\frac{\partial\mathbf b}{\partial z}
+
E\nabla^2\mathbf v,
$$

$$
\frac{\partial\mathbf b}{\partial t}
=
\frac{\partial\mathbf v}{\partial z}
+
E_m\nabla^2\mathbf b,
$$

with

$$
\nabla\cdot\mathbf v
=
\nabla\cdot\mathbf b
=
0.
$$

The dimensionless control parameters are the Lehnert number,

$$
Le
=
\frac{B_0}
{\Omega a\sqrt{\rho\mu_0}},
$$

the Ekman number,

$$
E
=
\frac{\nu}{\Omega a^2},
$$

and the magnetic Ekman number,

$$
E_m
=
\frac{\eta}{\Omega a^2}.
$$

Their ratio defines the magnetic Prandtl number,

$$
Pm
=
\frac{\nu}{\eta}
=
\frac{E}{E_m}.
$$

The dimensionless cylinder height is described by the aspect ratio

$$
\Gamma=\frac{h}{a},
$$

so that the computational domain is

$$
0\leq s\leq1,
\qquad
0\leq\varphi<2\pi,
\qquad
0\leq z\leq\Gamma.
$$

The Lehnert number measures the ratio of the rotation time to the Alfvén crossing time. The Ekman and magnetic Ekman numbers compare viscous and magnetic diffusion, respectively, with the rotation timescale.

---

### 1.2 Normal-mode representation

Because the basic state is invariant under rotations about the cylinder axis, different azimuthal Fourier modes evolve independently. We therefore seek perturbations proportional to

$$
e^{im\varphi-i\omega t},
\qquad
m\in\mathbb Z,
$$

where the eigenfrequency is generally complex:

$$
\omega=\omega_r+i\omega_i.
$$

With the adopted time dependence,

$$
e^{-i\omega t}
=
e^{-i\omega_rt}e^{\omega_it},
$$

so that stable dissipative modes satisfy

$$
\omega_i<0.
$$

The real part $\omega_r$ is the oscillation frequency, whereas

$$
-\omega_i
$$

is the damping rate.

The cylinder is closed by two endcaps. Standing waves are introduced in the axial direction, which quantizes the axial wavenumber as

$$
k=\frac{n\pi}{\Gamma},
\qquad
n\in\mathbb N.
$$

The horizontal and axial components have complementary sine and cosine dependences so that incompressibility is preserved. Each eigenmode is therefore identified by :

$$
(m,n),
$$

where $m$ is the azimuthal wavenumber, and $n$ is the axial mode number.

---

### 1.3 Toroidal-poloidal representation

The solenoidal constraints are satisfied identically by introducing toroidal and poloidal potentials relative to the uniform axial direction $\mathbf e_z$:

$$
\mathbf v
=
\nabla\times\left(T\mathbf e_z\right)
+
\nabla\times\nabla\times\left(P\mathbf e_z\right),
$$

$$
\mathbf b
=
\nabla\times\left(G\mathbf e_z\right)
+
\nabla\times\nabla\times\left(F\mathbf e_z\right).
$$

The scalar potentials are expanded as normal modes in $\varphi$, $z$, and $t$. Their radial parts are denoted by

$$
T(s),
\qquad
P(s),
\qquad
G(s),
\qquad
F(s).
$$

The axial direction is used as the decomposition direction because $\mathbf e_z$ is regular and spatially uniform throughout the cylinder, including at the rotation axis. This choice also permits the pressure to be eliminated by taking the axial components of the curl and double curl of the momentum equation.

The resulting problem consists of four coupled radial differential equations: two equations for the velocity potentials $T$ and $P$, and two equations for the magnetic potentials $G$ and $F$. Regularity at the axis and the physical conditions at the lateral boundary close the radial problem.

---

### 1.4 Limiting physical regimes

The formulation contains several important limits used to validate the numerical method.

In the non-magnetic ideal limit,

$$
Le\rightarrow0,
\qquad
E=E_m=0,
$$

the system reduces to the inviscid inertial-wave problem in a uniformly rotating cylinder. Its eigenfrequencies are the Kelvin--Greenspan frequencies.

For finite $Le$ with

$$
E=E_m=0,
$$

rotation and magnetic tension produce the ideal magneto-Coriolis modes. These provide a second analytical benchmark for the numerical eigenvalues.

When rotation becomes subdominant relative to magnetic tension, the fast branch approaches Alfvén-wave behaviour. Its leading frequency is proportional to

$$
\omega_A\sim Le\,k.
$$

Finally, for

$$
E>0,
\qquad
E_m>0,
$$

the eigenfrequencies become complex because of viscous and Ohmic dissipation. The ideal benchmarks then serve as reference states from which the corresponding dissipative modes are continued and identified.

In [ ]:
# === PACKAGES ===

import os
from pathlib import Path

# Must be set before importing NumPy/SciPy.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import sympy as sym

from IPython.display import display
from petsc4py import PETSc
from scipy.special import kv, kvp
from slepc4py import SLEPc

from mc_cylinder.cylindrical import div_cyl, simplify_vector
from mc_cylinder.fields import (
    horizontal_cartesian_components,
    magnetic,
    meridional_components,
    potential_radial_data,
    radial_component_profiles,
    velocity,
)
from mc_cylinder.greenspan import (
    greenspan_frequencies,
    ideal_magnetocoriolis_frequencies,
)
from mc_cylinder.lehnert import (
    lehnert_free_frequencies,
    lehnert_radial_mode,
    lehnert_radial_roots,
    solve_lehnert_with_general_mhd,
)
from mc_cylinder.symbolic import (
    compact_radial_mhd_residuals,
    derive_radial_mhd_equations,
    split_generalized_eigenvalue,
    toroidal_poloidal_ez,
)
from mc_cylinder.validation import (
    greenspan_benchmark,
    greenspan_convergence,
    ideal_mc_benchmark,
    solve_dissipative_mhd_mode,
    viscous_inertial_mode,
)

sym.init_printing()

## 2. Simulation parameters

In [ ]:
# === PHYSICAL MODEL ===

MODEL = dict(
    h=1.0,
    a=1.0,
    m=2,
    n_axial=1,
    B0=0.1,
    Omega_rot=1.0,
    rho=1.0,
    mu0=1.0,
    mu=1.0e-4,
    eta=1.0e-4,
)

NUMERICS = dict(
    N=36,
    nev=10,
    tol=1.0e-9,
)


def dimensionless_parameters(model):
    """Compute Le, E and Em from the dimensional model parameters."""
    a = model["a"]
    Omega = model["Omega_rot"]
    rho = model["rho"]
    mu0 = model["mu0"]

    Le = (
        model["B0"]
        / (Omega * a * np.sqrt(rho * mu0))
    )

    E = (
        model["mu"]
        / (rho * Omega * a**2)
    )

    Em = (
        model["eta"]
        / (Omega * a**2)
    )

    return {
        "Le": Le,
        "E": E,
        "Em": Em,
        "m": model["m"],
        "n_axial": model["n_axial"],
    }


# Automatically synchronized with MODEL.
PAPER_CASE = dimensionless_parameters(MODEL)

# One-based rank in the filtered dissipative spectrum.
PAPER_MODE_RANK = 1


print("Dimensionless parameters")
print(f"Le = {PAPER_CASE['Le']:.6e}")
print(f"E  = {PAPER_CASE['E']:.6e}")
print(f"Em = {PAPER_CASE['Em']:.6e}")


STUDY = dict(
    modes=[
        (1, 1), (2, 1), (3, 1),
        (1, 2), (2, 2), (3, 2),
    ],
    Le_values=np.logspace(-2, 1, 18),
    dissipative_cases=[
        (1.0e-7, 1.0e-2),
        (1.0e-4, 1.0e-3),
        (1.0e-2, 1.0e-2),
    ],
    N_development=36,
    N_final=36,
    N_convergence=[8, 12, 16, 20, 24, 28, 32, 36],
    N_reference=40,
)

## 3. Axial toroidal-poloidal decomposition and separation of variables

The velocity and magnetic perturbations are represented using the classical toroidal-poloidal decomposition relative to the uniform axial direction $\mathbf e_z$:

$$
\mathbf v
=
\nabla\times(T\mathbf e_z)
+
\nabla\times\nabla\times(P\mathbf e_z),
$$

$$
\mathbf b
=
\nabla\times(G\mathbf e_z)
+
\nabla\times\nabla\times(F\mathbf e_z).
$$

This representation satisfies the solenoidal constraints identically:

$$
\nabla\cdot\mathbf v=0,
\qquad
\nabla\cdot\mathbf b=0.
$$

For a single azimuthal Fourier mode, the dependence on $\varphi$ and time is taken to be

$$
\{T,P,G,F\}
\propto
e^{im\varphi-i\omega t}.
$$

To impose impermeability at the endcaps,

$$
v_z=0
\qquad
\text{at}
\qquad
z=0,\Gamma,
$$

we introduce standing waves with axial wavenumber

$$
k=\frac{n\pi}{\Gamma},
\qquad
n\in\mathbb N,
$$

and choose the complementary potentials

$$
T,F\propto\cos(kz),
\qquad
P,G\propto\sin(kz).
$$

This choice enforces

$$
v_z=0
\qquad
\text{at}
\qquad
z=0,\Gamma,
$$

but it does not generally impose

$$
v_s=v_\varphi=0
$$

at the endcaps. Therefore, throughout this notebook, the term *no-slip boundary condition* refers specifically to the lateral wall at $s=1$. A complete no-slip condition on both the lateral wall and the two endcaps cannot, in general, be represented by a single separated axial harmonic. Such a formulation would require a two-dimensional discretization in $(s,z)$.

With the standing-wave convention above, the physical velocity components are

\begin{align}
v_s
&=
\left(
kP'
+
\frac{im}{s}T
\right)
\cos(kz)e^{im\varphi},
\\
v_\varphi
&=
\left(
\frac{ikm}{s}P
-
T'
\right)
\cos(kz)e^{im\varphi},
\\
v_z
&=
-\mathcal L_mP
\sin(kz)e^{im\varphi}.
\end{align}

The magnetic components are

\begin{align}
b_s
&=
\left(
-kF'
+
\frac{im}{s}G
\right)
\sin(kz)e^{im\varphi},
\\
b_\varphi
&=
\left(
-\frac{ikm}{s}F
-
G'
\right)
\sin(kz)e^{im\varphi},
\\
b_z
&=
-\mathcal L_mF
\cos(kz)e^{im\varphi}.
\end{align}

Here, a prime denotes differentiation with respect to the dimensionless cylindrical radius $s$. The radial part of the horizontal Laplacian for the azimuthal mode $m$ is

$$
\mathcal L_m
=
\frac{d^2}{ds^2}
+
\frac{1}{s}\frac{d}{ds}
-
\frac{m^2}{s^2},
$$

and the radial part of the full three-dimensional Laplacian is

$$
\mathcal D_{m,k}
=
\mathcal L_m-k^2.
$$

Although these expressions contain apparently singular terms proportional to $1/s$ and $1/s^2$, the singularities are removable when the scalar potentials satisfy the correct pole regularity condition at the rotation axis. This regularization is introduced explicitly in the following section.

In [ ]:
s_sym, phi_sym, z_sym = sym.symbols("s phi z", positive=True, real=True)
m_sym = sym.symbols("m", integer=True, real=True)
k_sym = sym.symbols("k", positive=True, real=True)
phase_sym = sym.exp(sym.I*m_sym*phi_sym)
T_sym, P_sym, G_sym, F_sym = (sym.Function(name)(s_sym) for name in "TPGF")

v_sym = simplify_vector(toroidal_poloidal_ez(
    T_sym*phase_sym*sym.cos(k_sym*z_sym),
    P_sym*phase_sym*sym.sin(k_sym*z_sym), s_sym, phi_sym, z_sym,
))
b_sym = simplify_vector(toroidal_poloidal_ez(
    G_sym*phase_sym*sym.sin(k_sym*z_sym),
    F_sym*phase_sym*sym.cos(k_sym*z_sym), s_sym, phi_sym, z_sym,
))
assert sym.simplify(div_cyl(v_sym, s_sym, phi_sym, z_sym)) == 0
assert sym.simplify(div_cyl(b_sym, s_sym, phi_sym, z_sym)) == 0
display(sym.Matrix.hstack(sym.Matrix(v_sym), sym.Matrix(b_sym)))

## 4. Radial equations derived with SymPy

The pressure is eliminated from the momentum equation by taking its curl and double curl and projecting the resulting equations onto the axial direction $\mathbf e_z$. The induction equation and its axial curl provide the two corresponding magnetic equations.

After substitution of the toroidal--poloidal decomposition and the normal-mode dependence, the linear MHD system reduces to four coupled ordinary differential equations in the radial coordinate:

\begin{align}
i\omega T
+
E\mathcal D_{m,k}T
+
2kP
+
Le^2kG
&=0,
\\
-i\omega\mathcal D_{m,k}P
-
E\mathcal D_{m,k}^2P
-
2kT
+
Le^2k\mathcal D_{m,k}F
&=0,
\\
i\omega F
+
E_m\mathcal D_{m,k}F
+
kP
&=0,
\\
i\omega G
+
E_m\mathcal D_{m,k}G
-
kT
&=0.
\end{align}

The first two equations correspond respectively to the axial curl and axial double curl of the momentum equation. The last two equations arise from the poloidal and toroidal parts of the induction equation.

The radial operators are

$$
\mathcal L_m
=
\frac{d^2}{ds^2}
+
\frac{1}{s}\frac{d}{ds}
-
\frac{m^2}{s^2},
$$

and

$$
\mathcal D_{m,k}
=
\mathcal L_m-k^2.
$$

Introducing

$$
\mathbf q
=
\begin{bmatrix}
T\\
P\\
G\\
F
\end{bmatrix},
$$

the four residual equations may be separated into a frequency-independent part and a part proportional to $\omega$:

$$
\mathbf R(\mathbf q,\omega)
=
\mathbf A\mathbf q
-
\omega\mathbf B\mathbf q
=
\mathbf0.
$$

The continuous radial problem therefore has the generalized eigenvalue structure

$$
\mathbf A\mathbf q
=
\omega\mathbf B\mathbf q.
$$

In the following code cell, the four residuals are rederived directly from the dimensionless MHD equations using SymPy. Each residual is expanded and collected with respect to $\omega$, after which its frequency-independent and frequency-dependent contributions are extracted symbolically. This provides a term-by-term verification that the operators assembled numerically are identical to those obtained from the analytical derivation.

In [ ]:
omega_sym, Le_sym, E_sym, Em_sym = sym.symbols("omega Le E E_m", real=True)
radial_sym = derive_radial_mhd_equations(
    s=s_sym, phi=phi_sym, z=z_sym, m=m_sym, k=k_sym,
    omega=omega_sym, Le=Le_sym, E=E_sym, Em=Em_sym,
)
compact_sym = compact_radial_mhd_residuals(
    T=T_sym, P=P_sym, G=G_sym, F=F_sym, s=s_sym,
    m=m_sym, k=k_sym, omega=omega_sym, Le=Le_sym, E=E_sym, Em=Em_sym,
)
for derived, compact in zip(radial_sym.residuals, compact_sym):
    assert sym.expand(derived - compact) == 0
    A_part, B_part = split_generalized_eigenvalue(derived, omega_sym)
    assert sym.expand(derived + A_part - omega_sym*B_part) == 0
print("Four SymPy residuals match the compact equations and A q = omega B q.")

### Fully developped SymPy equations

In [ ]:
equation_names = (
    "Momentum — axial one curl",
    "Momentum — axial two curl",
    "Induction — axial zero curl",
    "Induction — axial one curl",
)

full_sympy_equations = []
generalized_sympy_blocks = []

for name, residual in zip(equation_names, radial_sym.residuals):
    expanded_residual = sym.collect(sym.expand(residual), omega_sym)
    A_part, B_part = split_generalized_eigenvalue(expanded_residual, omega_sym)
    full_sympy_equations.append(sym.Eq(expanded_residual, 0))
    generalized_sympy_blocks.append((sym.Eq(A_part, omega_sym*B_part), A_part, B_part))

    print(name)
    #display(sym.Eq(expanded_residual, 0))
    #print("Generalized eigenvalue form:")
    display(sym.Eq(A_part, omega_sym*B_part))

In [ ]:
latex_text = "\n\n".join(
    rf"""% {name}
\begin{{equation}}
\begin{{split}}
{sym.latex(equation)}
\end{{split}}
\end{{equation}}"""
    for name, equation in zip(
        equation_names,
        full_sympy_equations,
    )
)

print(latex_text)

## 5. Axis Regularity and Boundary Conditions

At the rotation axis, each regular potential is written as

$$
q(s)=s^{|m|}\widehat q(x),
\qquad
x=2s^2-1.
$$

This regularity is built directly into the discretization. At the lateral wall $s=1$, the no-slip condition gives

$$
\boxed{kP'+imT=0},
\qquad
\boxed{ikmP-T'=0},
\qquad
\boxed{\mathcal L_mP=0}.
$$

In the exterior vacuum,

$$
\Phi
=
C K_{|m|}(ks)e^{im\varphi}\sin(kz),
\qquad
\mathbf b_{\mathrm{ext}}=-\nabla\Phi.
$$

Eliminating the exterior amplitude $C$ yields

$$
\boxed{
imG-kF'
+\frac{K'_{|m|}(k)}{K_{|m|}(k)}
\mathcal L_mF=0
},
$$

$$
\boxed{
-G'-imkF
+\frac{im}{k}\mathcal L_mF=0
}.
$$

Two mechanical problems must be distinguished:

\begin{align}
E=0
&:\qquad v_s(1)=0
&&\text{(Kelvin-Greenspan interior problem)},\\
E>0
&:\qquad v_s(1)=v_\varphi(1)=v_z(1)=0
&&\text{(viscous no-slip problem)}.
\end{align}

### Regularity at the rotation axis

The cylindrical coordinates are singular at $s=0$, but the physical velocity and magnetic fields must remain single-valued and analytic in Cartesian coordinates. For a scalar Fourier component

$$
q(s,\varphi)=q_m(s)e^{im\varphi},
$$

analyticity at the axis requires the radial dependence to satisfy the pole condition

$$
q_m(s)
=
s^{|m|}
\left(
q_0+q_2s^2+q_4s^4+\cdots
\right)
=
s^{|m|}p(s^2).
$$

This is the regularity class used by Boroński and Tuckerman in their spectral formulation for a finite cylinder. They represent it with the Matsushima--Marcus radial polynomials $Q_n^m(s)$, whose admissible powers are

$$
s^{|m|},\qquad
s^{|m|+2},\qquad
s^{|m|+4},\ldots
$$

[Boroński & Tuckerman (2007)](https://doi.org/10.1016/j.jcp.2007.08.035). In the present formulation, the same analytic structure is enforced explicitly by writing

$$
q_m(s)=s^{|m|}\widehat q_m(x),
\qquad
x=2s^2-1,
$$

and expanding the regular function $\widehat q_m$ in Chebyshev polynomials,

$$
\widehat q_m(x)
=
\sum_{j=0}^{N-1}a_jT_j(x).
$$

Indeed,

$$
s^{|m|}T_j(2s^2-1)
=
s^{|m|}
\left(
c_0+c_2s^2+\cdots+c_{2j}s^{2j}
\right),
$$

so that the Chebyshev representation spans the same regularity class
$s^{|m|}p(s^2)$ as the Matsushima--Marcus basis. The two approaches therefore impose the same behaviour at the cylindrical axis, although they use different radial polynomial bases.

With the toroidal--poloidal decomposition taken along the axial direction $\mathbf e_z$, this condition applies identically to the four scalar potentials:

$$
T_m(s),\ P_m(s),\ G_m(s),\ F_m(s)
\in s^{|m|}\mathcal A(s^2),
$$

where $\mathcal A(s^2)$ denotes a function analytic in $s^2$. Hence, no distinction between the velocity and magnetic potentials is introduced at the axis. For $|m|\geq1$, all four potentials vanish as $s^{|m|}$. For the axisymmetric mode,

$$
m=0,
\qquad
q_0(s)=q_0+q_2s^2+q_4s^4+\cdots,
$$

and regularity instead implies the even-parity condition

$$
q_0'(0)=0.
$$

Possible additive gauge constants in the axisymmetric potentials are removed independently and should not be confused with physical regularity.

The factorization also removes the apparent singularity of the cylindrical radial Laplacian,

$$
\mathcal L_m
=
\frac{d^2}{ds^2}
+
\frac{1}{s}\frac{d}{ds}
-
\frac{m^2}{s^2}.
$$

Indeed,

$$
s^{-|m|}
\mathcal L_m
\left[
s^{|m|}\widehat q_m(x)
\right]
=
8(|m|+1)\frac{d\widehat q_m}{dx}
+
8(x+1)\frac{d^2\widehat q_m}{dx^2},
$$

which is regular throughout $-1\leq x\leq1$, including the rotation axis. Thus, axis regularity is incorporated into the approximation space itself rather than imposed through separate and potentially inconsistent boundary conditions on $T$, $P$, $G$, and $F$.

### Regularized field reconstruction and radial operators

For each azimuthal wavenumber $m$, let

$$
\ell=|m|,
\qquad
x=2s^2-1,
\qquad
-1\leq x\leq1.
$$

The four toroidal--poloidal potentials are written as

$$
T(s)=s^\ell\widehat T(x),
\qquad
P(s)=s^\ell\widehat P(x),
$$

$$
G(s)=s^\ell\widehat G(x),
\qquad
F(s)=s^\ell\widehat F(x).
$$

The regularized functions are expanded in the same Chebyshev basis,

$$
\widehat q(x)
=
\sum_{j=0}^{N-1}q_jT_j(x),
\qquad
q\in\{T,P,G,F\}.
$$

Because

$$
\frac{dx}{ds}=4s,
$$

the first radial derivative of a regularized scalar is

$$
\frac{dq}{ds}
=
s^{\ell-1}
\left[
\ell\widehat q
+
2(x+1)\frac{d\widehat q}{dx}
\right].
$$

Similarly,

$$
\frac{q}{s}
=
s^{\ell-1}\widehat q.
$$

The cylindrical radial Laplacian associated with the Fourier mode $m$ is

$$
\mathcal L_m q
=
\frac{d^2q}{ds^2}
+
\frac{1}{s}\frac{dq}{ds}
-
\frac{m^2}{s^2}q.
$$

After substitution of the regularized representation, the apparently singular terms cancel exactly:

$$
\mathcal L_m q
=
s^\ell
\left[
8(\ell+1)\frac{d\widehat q}{dx}
+
8(x+1)\frac{d^2\widehat q}{dx^2}
\right].
$$

We therefore introduce the regular operator

$$
\widehat{\mathcal L}_\ell
=
8(\ell+1)\frac{d}{dx}
+
8(x+1)\frac{d^2}{dx^2},
$$

such that

$$
\mathcal L_m
\left[
s^\ell\widehat q
\right]
=
s^\ell\widehat{\mathcal L}_\ell\widehat q.
$$

No division by zero is evaluated numerically at $s=0$: the limits of $q/s$, $q'$ and $\mathcal L_mq$ are obtained analytically from the regularized functions.

---

#### Physical velocity and magnetic components

The solenoidal fields are represented using the axial toroidal--poloidal decomposition

$$
\mathbf v
=
\nabla\times(T\mathbf e_z)
+
\nabla\times\nabla\times(P\mathbf e_z),
$$

$$
\mathbf b
=
\nabla\times(G\mathbf e_z)
+
\nabla\times\nabla\times(F\mathbf e_z).
$$

For a mode proportional to $e^{im\varphi}$ with axial wavenumber

$$
k=\frac{n\pi}{h},
$$

the complex radial amplitudes used in the numerical reconstruction are

$$
v_s
=
kP'
+
\frac{im}{s}T,
$$

$$
v_\varphi
=
\frac{ikm}{s}P
-
T',
$$

$$
v_z
=
-\mathcal L_mP,
$$

and

$$
b_s
=
-kF'
+
\frac{im}{s}G,
$$

$$
b_\varphi
=
-\frac{ikm}{s}F
-
G',
$$

$$
b_z
=
-\mathcal L_mF.
$$

The relative signs involving $k$ follow from the standing-wave convention adopted for the axial dependence. Substitution of the regularized potentials gives, for example,

$$
v_s
=
s^{\ell-1}
\left\{
k
\left[
\ell\widehat P
+
2(x+1)\widehat P_x
\right]
+
im\widehat T
\right\},
$$

$$
v_\varphi
=
s^{\ell-1}
\left\{
ikm\widehat P
-
\ell\widehat T
-
2(x+1)\widehat T_x
\right\},
$$

$$
v_z
=
-s^\ell
\widehat{\mathcal L}_\ell\widehat P.
$$

Likewise,

$$
b_s
=
s^{\ell-1}
\left\{
-k
\left[
\ell\widehat F
+
2(x+1)\widehat F_x
\right]
+
im\widehat G
\right\},
$$

$$
b_\varphi
=
s^{\ell-1}
\left\{
-ikm\widehat F
-
\ell\widehat G
-
2(x+1)\widehat G_x
\right\},
$$

$$
b_z
=
-s^\ell
\widehat{\mathcal L}_\ell\widehat F.
$$

These expressions demonstrate that the apparent factors $1/s$ in the cylindrical components do not constitute physical singularities. For $\ell\geq1$, the horizontal components behave at most as $s^{\ell-1}$ and the axial components as $s^\ell$, consistently with a Cartesian-analytic vector field.

For the axisymmetric case,

$$
m=0,
\qquad
\ell=0,
$$

the regularized potentials are even functions of $s$. Consequently,

$$
T'(0)=P'(0)=G'(0)=F'(0)=0,
$$

and therefore

$$
v_s(0)=v_\varphi(0)=b_s(0)=b_\varphi(0)=0.
$$

The axial components may remain finite:

$$
v_z(0)=-4P_{ss}(0),
\qquad
b_z(0)=-4F_{ss}(0),
$$

or, equivalently in terms of the coefficients of the local expansions,

$$
P(s)=p_0+p_2s^2+\mathcal O(s^4),
\qquad
v_z(0)=-4p_2,
$$

$$
F(s)=f_0+f_2s^2+\mathcal O(s^4),
\qquad
b_z(0)=-4f_2.
$$

This is the expected regular behaviour of an axisymmetric vector field at the rotation axis.

---

#### Lateral mechanical boundary conditions

At the lateral wall $s=1$, the viscous problem satisfies the complete no-slip condition

$$
\mathbf v(1)=\mathbf0.
$$

Using the reconstructed components, the three scalar conditions are

$$
kP'(1)+imT(1)=0,
$$

$$
ikmP(1)-T'(1)=0,
$$

$$
\mathcal L_mP(1)=0.
$$

These three coupled conditions enforce

$$
v_s(1)=v_\varphi(1)=v_z(1)=0.
$$

They are imposed only when viscosity is retained. In the inviscid Kelvin--Greenspan benchmark, the problem is of lower differential order and only impermeability is imposed:

$$
v_s(1)=0.
$$

Adding the two tangential no-slip conditions to the ideal problem would overconstrain the reduced inviscid eigenvalue problem.

---

#### Spectral discretization

Let $\mathbf D_x$ and $\mathbf D_x^{(2)}$ denote the Chebyshev differentiation matrices acting on the regularized functions. The transformed radial operator is represented by

$$
\widehat{\mathbf L}_\ell
=
8(\ell+1)\mathbf D_x
+
8\,\operatorname{diag}(x+1)\mathbf D_x^{(2)}.
$$

The four radial equations derived symbolically from the momentum and induction equations are then evaluated using the same regular operators for $T$, $P$, $G$, and $F$. After applying the physical boundary conditions, the discretized system takes the generalized eigenvalue form

$$
\mathbf A\mathbf q
=
\omega\mathbf B\mathbf q,
$$

with

$$
\mathbf q
=
\begin{bmatrix}
\widehat{\mathbf T}\\
\widehat{\mathbf P}\\
\widehat{\mathbf G}\\
\widehat{\mathbf F}
\end{bmatrix}.
$$

Thus, regularity is enforced before constructing the generalized eigenvalue problem: the matrices act on analytic unknowns $\widehat T$, $\widehat P$, $\widehat G$, and $\widehat F$, while the physical factors $s^{|m|}$ are restored only when reconstructing the velocity and magnetic fields.

In [ ]:
from mc_cylinder.discretization import (
    assemble_mhd_matrices,
    axisymmetric_gauge_fixed_basis,
)

s_reg = sym.symbols("s", positive=True, real=True)
k_reg = sym.symbols("k", positive=True, real=True)
q0, q2, q4 = sym.symbols("q_0 q_2 q_4")
t0, t2, p0, p2, g0, g2, f0, f2 = sym.symbols(
    "t_0 t_2 p_0 p_2 g_0 g_2 f_0 f_2"
)

def regular_series(ell, leading, correction):
    return s_reg**ell*(leading + correction*s_reg**2)

def Lm(expression, m_value):
    return sym.simplify(
        sym.diff(expression, s_reg, 2)
        + sym.diff(expression, s_reg)/s_reg
        - m_value**2*expression/s_reg**2
    )

axis_limits = []
for m_value in (0, 1, 2, 3):
    ell = abs(m_value)
    T_series = regular_series(ell, t0, t2)
    P_series = regular_series(ell, p0, p2)
    G_series = regular_series(ell, g0, g2)
    F_series = regular_series(ell, f0, f2)

    amplitudes = {
        "v_s": k_reg*sym.diff(P_series, s_reg) + sym.I*m_value*T_series/s_reg,
        "v_phi": sym.I*k_reg*m_value*P_series/s_reg - sym.diff(T_series, s_reg),
        "v_z": -Lm(P_series, m_value),
        "b_s": -k_reg*sym.diff(F_series, s_reg) + sym.I*m_value*G_series/s_reg,
        "b_phi": -sym.I*k_reg*m_value*F_series/s_reg - sym.diff(G_series, s_reg),
        "b_z": -Lm(F_series, m_value),
    }
    limits = {name: sym.limit(value, s_reg, 0, dir="+")
              for name, value in amplitudes.items()}
    assert all(not value.has(sym.oo, -sym.oo, sym.zoo) for value in limits.values())
    axis_limits.append((m_value, limits))

for m_value, limits in axis_limits:
    print(f"m = {m_value}")
    display(sym.Matrix([[sym.Symbol(name), value] for name, value in limits.items()]))

# Contrôle indépendant de la parité paire et d'une base sans constante.
axis_points = np.linspace(0.0, 1.0, 6)
Q0, Q0s = axisymmetric_gauge_fixed_basis(
    axis_points, modes=4, max_derivative=1
)
assert np.allclose(Q0[0], 0.0)   # q(0)=0 : jauge fixée
assert np.allclose(Q0s[0], 0.0)  # q'(0)=0 : parité paire
print("m=0: even parity and the gauge-fixed trial space are verified.")

# Le solveur conserve toutefois la base paire complète et fixe les jauges
# par quatre lignes explicites dans le pencil non divisé par L_0.
m0_pencil = assemble_mhd_matrices(
    size=12, m=0, k=np.pi, Le=PAPER_CASE["Le"],
    E=PAPER_CASE["E"], Em=PAPER_CASE["Em"],
    mechanical_boundary="no_slip",
)
assert len(m0_pencil.gauge_rows) == 4
assert np.linalg.matrix_rank(
    m0_pencil.A[list(m0_pencil.gauge_rows)]
) == 4
print("m=0: four independent gauge rows inserted into the MHD pencil.")

In [ ]:
x_reg = sym.symbols("x", real=True)
a0_reg, a1_reg, a2_reg = sym.symbols("a_0 a_1 a_2")
qhat_x = a0_reg + a1_reg*x_reg + a2_reg*x_reg**2
x_of_s = 2*s_reg**2 - 1

for m_value in (1, 2, 3):
    q_physical = s_reg**m_value*qhat_x.subs(x_reg, x_of_s)
    lhs = sym.expand(Lm(q_physical, m_value)/s_reg**m_value)
    rhs_x = (
        8*(m_value+1)*sym.diff(qhat_x, x_reg)
        + 8*(x_reg+1)*sym.diff(qhat_x, x_reg, 2)
    )
    rhs = sym.expand(rhs_x.subs(x_reg, x_of_s))
    assert sym.simplify(lhs-rhs) == 0

print("Regularized radial operator verified for m = 1, 2, 3.")

## 6. SLEPc solver

In [ ]:
# ---------------------------------------------------------------
# SLEPc for A u = ω B u 
# ---------------------------------------------------------------
def solve_evp_slepc(
    A,
    B,
    omega_target,
    nev=10,
    tol=1e-8,
    max_it=300,
    *,
    omega_ref=None,
    u_ref=None,
    imag_weight=0.1,
    omega_track_weight=0.2,
    overlap_weight=0.2,
):

    n = A.shape[0]
    assert A.shape == B.shape == (n, n)

    # -----------------------------------------------------------
    # Matrices PETSc
    # -----------------------------------------------------------
    A_p = PETSc.Mat().createAIJ(A.shape)
    B_p = PETSc.Mat().createAIJ(B.shape)
    A_p.setUp()
    B_p.setUp()

    rows = np.arange(n, dtype=np.int32)

    for i in range(n):
        A_p.setValues(i, rows, A[i, :])
        B_p.setValues(i, rows, B[i, :])

    A_p.assemble()
    B_p.assemble()

    # -----------------------------------------------------------
    # EPS solver
    # -----------------------------------------------------------
    E = SLEPc.EPS().create()
    E.setOperators(A_p, B_p)

    # General case
    E.setProblemType(SLEPc.EPS.ProblemType.GNHEP)

    # -----------------------------------------------------------
    # Dimensions du sous-espace
    # -----------------------------------------------------------
    ncv = max(4*nev, 20)
    mpd = ncv
    E.setDimensions(nev, ncv, mpd)

    # -----------------------------------------------------------
    # Ciblage d'une valeur propre réelle
    # -----------------------------------------------------------
    E.setTarget(omega_target)
    E.setWhichEigenpairs(SLEPc.EPS.Which.TARGET_REAL)

    # -----------------------------------------------------------
    # Shift-invert
    # -----------------------------------------------------------
    ST = E.getST()
    ST.setType(SLEPc.ST.Type.SINVERT)
    ST.setShift(omega_target)

    ksp = ST.getKSP()
    ksp.setType('preonly')

    pc = ksp.getPC()
    pc.setType('lu')
    pc.setFactorSolverType('mumps')

    # -----------------------------------------------------------
    # Tolérances 
    # -----------------------------------------------------------
    E.setTolerances(tol, max_it)
    E.setFromOptions()

    # -----------------------------------------------------------
    # Solve
    # -----------------------------------------------------------
    E.solve()

    nconv = E.getConverged()
    if nconv == 0:
        raise RuntimeError("SLEPc : aucune valeur propre convergée.")

    # -----------------------------------------------------------
    # Extraction des modes 
    # -----------------------------------------------------------
    omegas = np.zeros(nconv, dtype=complex)
    vecs   = np.zeros((n, nconv), dtype=complex)

    xr = PETSc.Vec().createSeq(n)
    xi = PETSc.Vec().createSeq(n)

    for i in range(nconv):
        lam = E.getEigenvalue(i)
        E.getEigenvector(i, xr, xi)
        omegas[i] = lam
        vecs[:, i] = xr.getArray() + 1j * xi.getArray()

    # -----------------------------------------------------------
    # Sélection du mode physique (proche de ω_target)
    # -----------------------------------------------------------
    score = np.abs(omegas.real - omega_target)
    # exp(-i*omega*t) => damping physique <=> Im(omega) < 0.
    score += imag_weight * np.maximum(omegas.imag, 0)

    if omega_ref is not None:
        score += omega_track_weight * np.abs(omegas - omega_ref)

    if u_ref is not None:
        u_ref_n = np.linalg.norm(u_ref)
        if u_ref_n > 0:
            overlaps = np.zeros(nconv)
            for i in range(nconv):
                vi = vecs[:, i]
                den = np.linalg.norm(vi) * u_ref_n
                if den > 0:
                    overlaps[i] = np.abs(np.vdot(u_ref, vi)) / den
            score += overlap_weight * (1.0 - overlaps)
    idx = np.argmin(score)

    omega_dom = omegas[idx]
    u_dom = vecs[:, idx].copy()

    # Normalisation simple
    umax = np.max(np.abs(u_dom))
    if umax > 0:
        u_dom /= umax

    return omega_dom, u_dom

## 7. Regularized Chebychev Assemble

In [ ]:
# === CHEBYSHEV MATRIX ===
def cheb_diff(N):
    if N == 0:
        return np.array([[0]]), np.array([1])
    x = -np.cos(np.pi * np.arange(N + 1) / N)
    c = np.ones(N + 1); c[0], c[-1] = 2, 2; c *= (-1) ** np.arange(N + 1)
    X = np.tile(x, (N + 1, 1)).T; dX = X - X.T + np.eye(N + 1)
    D = (np.outer(c, 1 / c)) / dX; D -= np.diag(np.sum(D, axis=1))
    s = 0.5 * (x + 1)
    return 2 * D, s

# === IDEAL MHD ===
def omega_MC_ideal(m, n, h, a, Omega_rot, Le, k):
    Gamma = h / a
    _, omega_I_values = greenspan_frequencies(
        m, n, Gamma, branch=1, count=1
    )
    omega_I = omega_I_values[0]
    k_dimensionless = n * np.pi / Gamma
    omega_fast, _ = ideal_magnetocoriolis_frequencies(
        omega_I, k_dimensionless, Le
    )
    return Omega_rot * omega_fast

def solve_mhd_evp(
    N,
    *,
    h,
    a,
    m,
    n_axial,
    B0,
    Omega_rot,
    rho,
    mu0,
    mu,
    eta,
    mechanical_boundary="auto",
    nev=6,
    tol=1e-8,
    prev_omega=None,
    prev_solution=None,
):
    # -------------------
    # Derived dimensionless parameters (length a, time Omega_rot**-1)
    # -------------------
    Gamma = h / a
    k = n_axial * np.pi / Gamma
    nu = mu / rho
    alfven_speed = B0 / np.sqrt(mu0 * rho)
    Le = alfven_speed / (Omega_rot * a)
    E = nu / (Omega_rot * a**2)
    Em = eta / (Omega_rot * a**2)
    if mechanical_boundary == "auto":
        mechanical_boundary = "inviscid" if np.isclose(E, 0.0) else "no_slip"
    if mechanical_boundary not in {"inviscid", "no_slip"}:
        raise ValueError("mechanical_boundary must be 'auto', 'inviscid', or 'no_slip'")
    if mechanical_boundary == "inviscid" and not np.isclose(E, 0.0):
        raise ValueError("the inviscid boundary is only consistent when E=0")
    if mechanical_boundary == "no_slip" and np.isclose(E, 0.0):
        raise ValueError("strict no-slip requires E>0; use the inviscid boundary when E=0")
    omega_MC = omega_MC_ideal(
        m, n_axial, h, a, Omega_rot, Le, k
    )
    omega_target = omega_MC / Omega_rot

    # -------------------
    # New SymPy-derived e_z equations.  We write every regular potential as
    # q(s)=s**|m| qhat(x), x=2*s**2-1.  This removes the apparent 1/s
    # singularities without changing the Chebyshev/SLEPc workflow.
    # -------------------
    n_p = N + 1
    Dx_linear, _ = cheb_diff(N)
    Dx = 0.5 * Dx_linear
    x = -np.cos(np.pi * np.arange(n_p) / N)
    s = np.sqrt(0.5 * (x + 1.0))
    ell = abs(m)
    I = np.eye(n_p, dtype=complex)
    R = 8.0 * (ell + 1) * Dx + 8.0 * np.diag(x + 1.0) @ (Dx @ Dx)
    Dop = R - k**2 * I

    size = 4 * n_p
    A = np.zeros((size, size), dtype=complex)
    B = np.zeros_like(A)
    T_idx = np.arange(0, n_p)
    P_idx = np.arange(n_p, 2*n_p)
    G_idx = np.arange(2*n_p, 3*n_p)
    F_idx = np.arange(3*n_p, 4*n_p)

    # A u = omega B u is the matrix form of the four compact radial equations.
    A[np.ix_(T_idx, T_idx)] = -E * Dop
    A[np.ix_(T_idx, P_idx)] = -2.0 * k * I
    A[np.ix_(T_idx, G_idx)] = -(Le**2) * k * I
    B[np.ix_(T_idx, T_idx)] = 1j * I

    A[np.ix_(P_idx, T_idx)] = 2.0 * k * I
    A[np.ix_(P_idx, P_idx)] = E * (Dop @ Dop)
    A[np.ix_(P_idx, F_idx)] = -(Le**2) * k * Dop
    B[np.ix_(P_idx, P_idx)] = -1j * Dop

    A[np.ix_(G_idx, P_idx)] = -k * I
    A[np.ix_(G_idx, F_idx)] = -Em * Dop
    B[np.ix_(G_idx, F_idx)] = 1j * I

    A[np.ix_(F_idx, T_idx)] = k * I
    A[np.ix_(F_idx, G_idx)] = -Em * Dop
    B[np.ix_(F_idx, G_idx)] = 1j * I

    # Wall functionals at s=1 for q=s**ell*qhat(x).
    value = I[-1]
    derivative = ell * value + 4.0 * Dx[-1]
    laplacian = R[-1]
    vacuum_ratio = kvp(ell, k) / kv(ell, k)

    # The differential order drops in the singular ideal limits.  Impose only
    # the boundary data belonging to the remaining differential operators.
    mechanical_rows = (
        (T_idx[-1], P_idx[-1], P_idx[-2])
        if mechanical_boundary == "no_slip" else (T_idx[-1],)
    )
    magnetic_rows = (G_idx[-1], F_idx[-1]) if not np.isclose(Em, 0.0) else ()
    boundary_rows = mechanical_rows + magnetic_rows
    for row in boundary_rows:
        A[row, :] = 0.0
        B[row, :] = 0.0
    # Impermeability is common to both mechanical problems.
    A[mechanical_rows[0], T_idx] = 1j*m*value
    A[mechanical_rows[0], P_idx] = k*derivative
    if mechanical_boundary == "no_slip":
        A[mechanical_rows[1], T_idx] = -derivative
        A[mechanical_rows[1], P_idx] = 1j*k*m*value
        A[mechanical_rows[2], P_idx] = laplacian
    if magnetic_rows:
        A[magnetic_rows[0], G_idx] = 1j*m*value
        A[magnetic_rows[0], F_idx] = -k*derivative + vacuum_ratio*laplacian
        A[magnetic_rows[1], G_idx] = -derivative
        A[magnetic_rows[1], F_idx] = -1j*m*k*value + (1j*m/k)*laplacian

    # Use one coefficient-space assembler for every m. This is essential:
    # the reconstruction below evaluates Chebyshev coefficients, not nodal
    # collocation values. For m=0 the assembler additionally retains L_0
    # and inserts the four explicit gauge rows.
    gauge_rows = ()
    from mc_cylinder.discretization import (
        assemble_ideal_mhd_matrices, assemble_mhd_matrices,
    )
    if np.isclose(E, 0.0) and np.isclose(Em, 0.0):
        problem = assemble_ideal_mhd_matrices(
            size=n_p, m=m, k=k, Le=Le,
        )
    elif E > 0.0 and Em > 0.0:
        problem = assemble_mhd_matrices(
            size=n_p, m=m, k=k, Le=Le, E=E, Em=Em,
            mechanical_boundary=mechanical_boundary,
        )
    else:
        raise ValueError(
            "Use either E=Em=0 or the fully dissipative case E>0, Em>0."
        )
    A, B = problem.A, problem.B
    boundary_rows = problem.boundary_rows
    gauge_rows = problem.gauge_rows

    # Keep the original SLEPc workflow and target/overlap selection.
    omega_ref = None if prev_omega is None else prev_omega / Omega_rot
    omega_nd, coefficients = solve_evp_slepc(
        A,
        B,
        omega_target,
        nev=nev,
        tol=tol,
        omega_ref=omega_ref,
        u_ref=prev_solution,
    )

    residual_vector = A @ coefficients - omega_nd * (B @ coefficients)
    residual = np.linalg.norm(residual_vector) / (
        (np.linalg.norm(A, ord="fro")
         + abs(omega_nd) * np.linalg.norm(B, ord="fro"))
        * np.linalg.norm(coefficients)
    )

    return {
        "omega": Omega_rot * omega_nd,
        "residual": residual,
        "s": s,
        # These are the regular amplitudes qhat; q=s**|m| qhat.
        "solution": coefficients,
        "regularized": True,
        "radial_representation": "chebyshev_coefficients",
        "indices": (T_idx, P_idx, G_idx, F_idx),
        "omega_MC": omega_MC,
        "Omega_rot": Omega_rot,
        "Gamma": Gamma,
        "Le": Le,
        "E": E,
        "Em": Em,
        "mechanical_boundary": mechanical_boundary,
        "boundary_rows": boundary_rows,
        "gauge_rows": gauge_rows,
    }

## 8. Computation

In [ ]:
out = solve_mhd_evp(**NUMERICS, **MODEL)
print("omega_MC =", out["omega_MC"])
print("omega_num =", out["omega"])
print("residual =", out["residual"])

### Kelvin–Greenspan Validation

This validation is independent of the four-potential MHD formulation. It directly solves the inviscid hydrodynamic problem in terms of $(T,P)$, with $E=Le=0$ and the only lateral boundary condition compatible with the reduced differential order,

$$
v_s(1)=0.
$$

Using the convention

$$
\exp\left[i(m\varphi+kz-\omega t)\right]
$$

adopted throughout this notebook, the positive-frequency branch can be written in the factorized form

$$
\boxed{
\omega_I
=
2\left(
1+\frac{\xi^2h^2}{n^2\pi^2a^2}
\right)^{-1/2}
},
$$

where $\xi$ satisfies

$$
\boxed{
\xi J_{|m|}'(\xi)
-
m
\left(
1+\frac{\xi^2h^2}{n^2\pi^2a^2}
\right)^{1/2}
J_{|m|}(\xi)
=
0
}.
$$

The opposite-sign branch is also present in the complete spectrum.

This comparison does not yet validate the viscous no-slip problem. Its limit as $E\rightarrow0$ is singular because of the lateral Ekman boundary layer.

In [ ]:
GREENSPAN_MODES = ((1, 1), (2, 1), (3, 1), (1, 2), (2, 2), (3, 2))
GREENSPAN_N = (8,16, 20, 24, 28, 32)
GREENSPAN_RADIAL_COUNT = 1

greenspan_rows = tuple(
    record
    for m, n in GREENSPAN_MODES
    for record in greenspan_benchmark(
        m=m, n=n, Gamma=MODEL["h"]/MODEL["a"],
        resolution=GREENSPAN_N[-1], radial_count=GREENSPAN_RADIAL_COUNT,
        branches=(1,),
    )
)

print(f"{len(greenspan_rows)} analytical/numerical eigenfrequency matches")
print("maximum absolute error =",
      max(row.absolute_error for row in greenspan_rows))
print("maximum backward residual =",
      max(row.backward_residual for row in greenspan_rows))
print("\n m  n  j  omega_I exact   omega_I numerical  absolute error")
for row in greenspan_rows:
    print(f"{row.m:2d} {row.n:2d} {row.radial_index:2d}  "
          f"{row.exact_frequency:+.10f}  {row.numerical_frequency.real:+.10f}  "
          f"{row.absolute_error:.3e}")

### Validation ideal magnetocoriolis

We now retain $Le>0$ while setting $E=E_m=0$. For each inertial frequency validated in the previous section, the ideal induction equation provides an algebraic relation between the magnetic and mechanical potentials.

$$
\boxed{
\omega_{\mathrm{MC}}^\pm
=
\frac{\Omega\omega_I}{2}
\left[
1
\pm
\left(
1+\frac{4Le^2k^2}
{\Omega^2\omega_I^2}
\right)^{1/2}
\right]
}.
$$

The numerical matrix is assembled using all four unknowns $(T,P,G,F)$.

In [ ]:
IDEAL_MC_MODES = GREENSPAN_MODES
IDEAL_MC_N = 32
IDEAL_MC_LE = 0.1
IDEAL_MC_RADIAL_COUNT = 1

ideal_mc_rows = tuple(
    record
    for m, n in IDEAL_MC_MODES
    for record in ideal_mc_benchmark(
        m=m,
        n=n,
        Gamma=MODEL["h"] / MODEL["a"],
        Le=IDEAL_MC_LE,
        resolution=IDEAL_MC_N,
        radial_count=IDEAL_MC_RADIAL_COUNT,
        inertial_branches=(1,),
    )
    if record.mc_branch == "fast"
)

print(f"{len(ideal_mc_rows)} analytical/numerical fast-MC matches")
print(
    "maximum absolute error =",
    max(row.absolute_error for row in ideal_mc_rows),
)
print(
    "maximum backward residual =",
    max(row.backward_residual for row in ideal_mc_rows),
)
print("\n m  n  j  omega_MC exact  omega_MC numerical  absolute error")
for row in ideal_mc_rows:
    print(
        f"{row.m:2d} {row.n:2d} {row.radial_index:2d}  "
        f"{row.exact_frequency:+.10f}  "
        f"{row.numerical_frequency.real:+.10f}  "
        f"{row.absolute_error:.3e}"
    )


## 9. Validation of the Complete Viscous–Diffusive MHD Problem

When $E>0$ and $E_m>0$, the three no-slip conditions and the two vacuum-matching conditions are active. No exact algebraic expression is available for the eigenfrequencies of the complete bounded problem. Its validation therefore relies simultaneously on:

1. the convergence of $\Re\omega$ and $\Im\omega$ with the radial resolution $N$;
2. the backward residual of $Aq=\omega Bq$;
3. the five lateral boundary residuals;
4. the overlap between the fields computed at successive resolutions.

The overlap is evaluated after reconstructing $(T,P,G,F)$ on a common radial grid. Its absolute value is independent of the arbitrary complex phase of the eigenvector. A value close to unity confirms that the same eigenmode is followed between successive resolutions, even in the presence of nearby eigenvalues.

In [ ]:
# === FULL VISCOUS–DIFFUSIVE MHD VALIDATION: E > 0, Em > 0 ===

FULL_MHD_MODES = (
    (1, 1), (2, 1), (3, 1),
    (1, 2), (2, 2), (3, 2),
)

FULL_MHD_N = (8, 12, 20, 28, 32, 48, 92)

FULL_MHD_LE = PAPER_CASE["Le"]
FULL_MHD_E = PAPER_CASE["E"]
FULL_MHD_EM = PAPER_CASE["Em"]

Gamma = MODEL["h"] / MODEL["a"]

assert FULL_MHD_E > 0.0
assert FULL_MHD_EM > 0.0


def track_resolution(mode):
    """Follow the same physical MC mode while increasing N."""

    m, n = mode
    k = n * np.pi / Gamma

    # Ideal MC frequency used only as the initial spectral target.
    omega_I = greenspan_frequencies(
        m,
        n,
        Gamma,
        branch=1,
        count=1,
    )[1][0]

    omega_fast, _ = ideal_magnetocoriolis_frequencies(
        omega_I,
        k,
        FULL_MHD_LE,
    )

    target = omega_fast
    previous_fingerprint = None
    rows = []

    for N in FULL_MHD_N:
        result = solve_dissipative_mhd_mode(
            resolution=N,
            m=m,
            k=k,
            Le=FULL_MHD_LE,
            E=FULL_MHD_E,
            Em=FULL_MHD_EM,
            target=target,
            previous_fingerprint=previous_fingerprint,
            mechanical_boundary="no_slip",
        )

        rows.append(result)

        # Continue from the mode found at the preceding resolution.
        target = result.frequency
        previous_fingerprint = result.fingerprint

    return rows


full_mhd_rows = {
    mode: track_resolution(mode)
    for mode in FULL_MHD_MODES
}


# === NUMERICAL DIAGNOSTICS ===

for mode, rows in full_mhd_rows.items():
    overlaps = [
        row.overlap_with_previous
        for row in rows[1:]
        if row.overlap_with_previous is not None
    ]
    """
    print(
        f"{mode}: "
        f"omega = {rows[-1].frequency:+.10e}, "
        f"minimum overlap = {min(overlaps):.6f}, "
        f"maximum wall residual = "
        f"{max(row.boundary_residual for row in rows):.3e}, "
        f"maximum backward residual = "
        f"{max(row.backward_residual for row in rows):.3e}"
    )
    """
# Orange for n=1, blue for n=2.

colors_n1 = plt.cm.Oranges(np.linspace(0.45, 0.90, 3))
colors_n2 = plt.cm.Blues(np.linspace(0.45, 0.90, 3))

mode_colors = {
    (1, 1): colors_n1[0],
    (2, 1): colors_n1[1],
    (3, 1): colors_n1[2],
    (1, 2): colors_n2[0],
    (2, 2): colors_n2[1],
    (3, 2): colors_n2[2],
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    constrained_layout=True,
    sharex=True,
)

N_plot = np.asarray(FULL_MHD_N[:-1])
N_reference = FULL_MHD_N[-1]

for mode, rows in full_mhd_rows.items():
    frequencies = np.asarray([
        row.frequency
        for row in rows
    ])

    reference = frequencies[-1]
    color = mode_colors[mode]

    real_errors = np.abs(
        frequencies[:-1].real - reference.real
    )

    imag_errors = np.abs(
        frequencies[:-1].imag - reference.imag
    )

    axes[0].semilogy(
        N_plot,
        real_errors,
        "o-",
        color=color,
        linewidth=1.5,
        markersize=4,
        label=fr"$(m,n)={mode}$",
    )

    axes[1].semilogy(
        N_plot,
        imag_errors,
        "o-",
        color=color,
        linewidth=1.5,
        markersize=4,
        label=fr"$(m,n)={mode}$",
    )

axes[0].set(
    xlabel=r"radial resolution $N$",
    ylabel=fr"$|\Re(\omega_N)-\Re(\omega_{{{N_reference}}})|$",
    title="Oscillation frequency",
)

axes[1].set(
    xlabel=r"radial resolution $N$",
    ylabel=fr"$|\Im(\omega_N)-\Im(\omega_{{{N_reference}}})|$",
    title="Damping rate",
)

for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(
        frameon=False,
        ncol=2,
        fontsize=10,
    )

fig.savefig(OUTPUT_DIR / "diffusive_convergence.png", dpi=300, bbox_inches="tight")
plt.show()


## 10. Fields Reconstruction

In [ ]:
# === SINGLE SELECTED MHD MODE ===

from matplotlib.patches import Circle


def reconstruct_horizontal_slice(
    out,
    *,
    m,
    n_axial,
    z_fraction=0.30,
    tau=0.0,
    Ncart=500,
):
    """Reconstruct one horizontal slice of a computed eigenmode."""

    Gamma = out["Gamma"]
    k = n_axial * np.pi / Gamma

    common = dict(
        solution=out["solution"],
        indices=out["indices"],
        m=m,
        k=k,
        omega=out["omega"] / out["Omega_rot"],
        z=z_fraction * Gamma,
        time=tau,
        resolution=Ncart,
    )

    x, y, XC, YC, vx, vy, vz = horizontal_cartesian_components(
        **common,
        kind="velocity",
    )

    _, _, _, _, bx, by, bz = horizontal_cartesian_components(
        **common,
        kind="magnetic",
    )

    # Normalize velocity and magnetic fields independently because the
    # eigenvector amplitude is arbitrary.
    velocity_amplitude = np.sqrt(
        np.abs(vx)**2
        + np.abs(vy)**2
        + np.abs(vz)**2
    )

    magnetic_amplitude = np.sqrt(
        np.abs(bx)**2
        + np.abs(by)**2
        + np.abs(bz)**2
    )

    velocity_norm = np.nanmax(velocity_amplitude)
    magnetic_norm = np.nanmax(magnetic_amplitude)

    if not np.isfinite(velocity_norm) or velocity_norm == 0.0:
        velocity_norm = 1.0

    if not np.isfinite(magnetic_norm) or magnetic_norm == 0.0:
        magnetic_norm = 1.0

    return {
        "x": x,
        "y": y,
        "XC": XC,
        "YC": YC,
        "vx": vx / velocity_norm,
        "vy": vy / velocity_norm,
        "vz": vz / velocity_norm,
        "bx": bx / magnetic_norm,
        "by": by / magnetic_norm,
        "bz": bz / magnetic_norm,
    }


def solve_and_plot_single_mode(
    *,
    E,
    Em,
    m,
    n_axial,
    radial_index=1,
    mc_branch="fast",
    inertial_branch=1,
    N=None,
    nev=None,
    tol=None,
    z_fraction=0.30,
    tau=0.0,
    Ncart=500,
):
    """
    Solve and display a single MHD case with user-selected E and Em.

    The corresponding ideal MC frequency is used only as a spectral target;
    no ideal comparison is plotted.
    """

    if E < 0.0 or Em < 0.0:
        raise ValueError("E and Em must be non-negative.")

    if radial_index < 1:
        raise ValueError("radial_index must be at least one.")

    if mc_branch not in {"fast", "slow"}:
        raise ValueError("mc_branch must be 'fast' or 'slow'.")

    N = NUMERICS["N"] if N is None else N
    nev = NUMERICS["nev"] if nev is None else nev
    tol = NUMERICS["tol"] if tol is None else tol

    Gamma = MODEL["h"] / MODEL["a"]
    k = n_axial * np.pi / Gamma

    # --------------------------------------------------------------
    # Convert dimensionless diffusivities into MODEL parameters
    # --------------------------------------------------------------

    selected_model = dict(
        MODEL,
        m=m,
        n_axial=n_axial,
        mu=(
            E
            * MODEL["rho"]
            * MODEL["Omega_rot"]
            * MODEL["a"]**2
        ),
        eta=(
            Em
            * MODEL["Omega_rot"]
            * MODEL["a"]**2
        ),
    )

    # --------------------------------------------------------------
    # Analytical ideal MC target used only for mode selection
    # --------------------------------------------------------------

    _, inertial_frequencies = greenspan_frequencies(
        m,
        n_axial,
        Gamma,
        branch=inertial_branch,
        count=radial_index,
    )

    omega_I = inertial_frequencies[radial_index - 1]

    Le = (
        selected_model["B0"]
        / (
            selected_model["Omega_rot"]
            * selected_model["a"]
            * np.sqrt(
                selected_model["rho"]
                * selected_model["mu0"]
            )
        )
    )

    omega_fast, omega_slow = ideal_magnetocoriolis_frequencies(
        omega_I,
        k,
        Le,
    )

    target = (
        omega_fast
        if mc_branch == "fast"
        else omega_slow
    )

    # --------------------------------------------------------------
    # Solve only the selected case
    # --------------------------------------------------------------

    out = solve_mhd_evp(
        N=N,
        nev=nev,
        tol=tol,
        prev_omega=target,
        **selected_model,
    )

    fields = reconstruct_horizontal_slice(
        out,
        m=m,
        n_axial=n_axial,
        z_fraction=z_fraction,
        tau=tau,
        Ncart=Ncart,
    )

    # --------------------------------------------------------------
    # Figure
    # --------------------------------------------------------------

    velocity_scale = np.nanmax(np.abs(fields["vz"]))
    magnetic_scale = np.nanmax(np.abs(fields["bz"]))

    if not np.isfinite(velocity_scale) or velocity_scale == 0.0:
        velocity_scale = 1.0

    if not np.isfinite(magnetic_scale) or magnetic_scale == 0.0:
        magnetic_scale = 1.0

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11.5, 5.2),
        constrained_layout=True,
    )

    panels = (
        (
            axes[0],
            fields["vz"],
            fields["vx"],
            fields["vy"],
            "Blues",
            velocity_scale,
            "Velocity",
            r"$v_z/\max|\mathbf{v}|$",
        ),
        (
            axes[1],
            fields["bz"],
            fields["bx"],
            fields["by"],
            "Oranges",
            magnetic_scale,
            "Magnetic field",
            r"$b_z/\max|\mathbf{b}|$",
        ),
    )

    for (
        ax,
        vertical,
        horizontal_x,
        horizontal_y,
        cmap,
        scale,
        title,
        colorbar_label,
    ) in panels:

        mesh = ax.pcolormesh(
            fields["XC"],
            fields["YC"],
            vertical,
            shading="auto",
            cmap=cmap,
            vmin=-scale,
            vmax=scale,
            rasterized=True,
        )

        ax.streamplot(
            fields["x"],
            fields["y"],
            horizontal_x,
            horizontal_y,
            color="black",
            density=1.25,
            linewidth=0.65,
            arrowsize=0.8,
        )

        ax.add_patch(
            Circle(
                (0.0, 0.0),
                1.0,
                fill=False,
                color="black",
                linewidth=1.0,
                zorder=10,
            )
        )

        colorbar = fig.colorbar(
            mesh,
            ax=ax,
            fraction=0.046,
            pad=0.025,
        )

        colorbar.set_label(colorbar_label)

        ax.set(
            title=title,
            aspect="equal",
            xlim=(-1.02, 1.02),
            ylim=(-1.02, 1.02),
        )

        ax.set_axis_off()

    regime = (
        "ideal"
        if E == 0.0 and Em == 0.0
        else "viscous"
        if E > 0.0 and Em == 0.0
        else "resistive"
        if E == 0.0 and Em > 0.0
        else "viscous–diffusive"
    )
    """
    fig.suptitle(
        fr"{regime.capitalize()} mode $(m,n,j)"
        fr"=({m},{n_axial},{radial_index})$, "
        fr"$E={E:.1e}$, $E_m={Em:.1e}$, "
        fr"$z/h={z_fraction:.2f}$",
        fontsize=13,
    )
    
    print(f"regime              = {regime}")
    print(f"(m, n, j)           = ({m}, {n_axial}, {radial_index})")
    print(f"MC branch           = {mc_branch}")
    print(f"Le                  = {Le:.6e}")
    print(f"E                   = {E:.6e}")
    print(f"Em                  = {Em:.6e}")
    print(f"ideal target        = {target:+.10e}")
    print(f"computed frequency  = {out['omega']:+.10e}")
    print(f"damping rate        = {-out['omega'].imag:.10e}")
    """
    return out, fig, axes

In [ ]:
def plot_near_wall_profiles(
    out,
    *,
    m,
    n_axial,
    E,
    Em,
    radial_resolution=1001,
    radial_min=0.80,
    amplitude_floor=1.0e-14,
):
    """
    Plot normalized physical-component envelopes near the lateral wall.

    Velocity and magnetic fields are normalized independently.
    """

    Gamma = out["Gamma"]
    k = n_axial * np.pi / Gamma

    # Chebyshev-like grid clustered near s=0 and s=1.
    theta = np.linspace(0.0, np.pi, radial_resolution)
    s_values = 0.5 * (1.0 - np.cos(theta))

    vs_amp, vphi_amp, vz_amp = radial_component_profiles(
        out["solution"],
        s_values,
        out["indices"],
        m,
        k,
        kind="velocity",
    )

    bs_amp, bphi_amp, bz_amp = radial_component_profiles(
        out["solution"],
        s_values,
        out["indices"],
        m,
        k,
        kind="magnetic",
    )

    # Phase-independent component envelopes.
    velocity_profiles = np.abs(
        np.vstack((vs_amp, vphi_amp, vz_amp))
    )

    magnetic_profiles = np.abs(
        np.vstack((bs_amp, bphi_amp, bz_amp))
    )

    # Independent normalizations.
    velocity_scale = np.nanmax(velocity_profiles)
    magnetic_scale = np.nanmax(magnetic_profiles)

    if not np.isfinite(velocity_scale) or velocity_scale == 0.0:
        velocity_scale = 1.0

    if not np.isfinite(magnetic_scale) or magnetic_scale == 0.0:
        magnetic_scale = 1.0

    velocity_profiles /= velocity_scale
    magnetic_profiles /= magnetic_scale

    labels_velocity = (
        r"$|v_s|$",
        r"$|v_\varphi|$",
        r"$|v_z|$",
    )

    labels_magnetic = (
        r"$|b_s|$",
        r"$|b_\varphi|$",
        r"$|b_z|$",
    )

    colors = (
        "#2166AC",
        "#D95F02",
        "#1B9E77",
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 4),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    for profile, label, color in zip(
        velocity_profiles,
        labels_velocity,
        colors,
    ):
        axes[0].semilogy(
            s_values,
            np.maximum(profile, amplitude_floor),
            color=color,
            linewidth=1.8,
            label=label,
        )

    for profile, label, color in zip(
        magnetic_profiles,
        labels_magnetic,
        colors,
    ):
        axes[1].semilogy(
            s_values,
            np.maximum(profile, amplitude_floor),
            color=color,
            linewidth=1.8,
            label=label,
        )

    # Wall position.
    for ax in axes:
        ax.axvline(
            1.0,
            color="black",
            linestyle="--",
            linewidth=1.0,
        )

        ax.set(
            xlabel=r"$s/a$",
            xlim=(radial_min, 1.002),
            ylim=(amplitude_floor, 1.2),
        )

        ax.grid(
            True,
            which="both",
            alpha=0.25,
        )

        ax.legend(
            frameon=False,
            fontsize=9,
        )

    axes[0].set_ylabel(
        "normalized component amplitude"
    )

    if E > 0.0:
        axes[0].set_title(
            r"No-slip wall: $\mathbf{v}(1)=0$"
        )
    else:
        axes[0].set_title(
            r"Inviscid wall: $v_s(1)=0$ only"
        )

    if Em > 0.0:
        axes[1].set_title(
            r"Vacuum matching: $\mathbf{b}(1)$"
        )
    else:
        axes[1].set_title(
            "Ideal magnetic problem"
        )
    """
    fig.suptitle(
        fr"Near-wall profiles for $(m,n)=({m},{n_axial})$, "
        fr"$E={E:.1e}$, $E_m={Em:.1e}$",
        fontsize=12,
    )
    """
    # Values at s=1, evaluated directly from the reconstruction.
    wall_velocity = velocity_profiles[:, -1]
    wall_magnetic = magnetic_profiles[:, -1]

    print("Normalized component amplitudes at s = 1")
    print()
    print("Velocity")
    print(f"|v_s|       = {wall_velocity[0]:.3e}")
    print(f"|v_phi|     = {wall_velocity[1]:.3e}")
    print(f"|v_z|       = {wall_velocity[2]:.3e}")

    print()
    print("Magnetic field")
    print(f"|b_s|       = {wall_magnetic[0]:.3e}")
    print(f"|b_phi|     = {wall_magnetic[1]:.3e}")
    print(f"|b_z|       = {wall_magnetic[2]:.3e}")

    return fig, axes, {
        "s": s_values,
        "velocity_profiles": velocity_profiles,
        "magnetic_profiles": magnetic_profiles,
        "wall_velocity": wall_velocity,
        "wall_magnetic": wall_magnetic,
    }

In [ ]:
CASE_E = 1.0e-4
CASE_EM = 1.0e-4
CASE_M = 2
CASE_N = 1

out, fig_fields, axes_fields = solve_and_plot_single_mode(
    E=CASE_E,
    Em=CASE_EM,
    m=CASE_M,
    n_axial=CASE_N,
    radial_index=1,
    mc_branch="fast",
    N=92,
    z_fraction=0.30,
)

fig_wall, axes_wall, wall_diagnostics = plot_near_wall_profiles(
    out,
    m=CASE_M,
    n_axial=CASE_N,
    E=CASE_E,
    Em=CASE_EM,
)

fig_fields.savefig(OUTPUT_DIR / "Fields.png", dpi=300, bbox_inches="tight")
fig_wall.savefig(OUTPUT_DIR / "BC_conditions.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def plot_endcap_profiles(
    out,
    *,
    m,
    n_axial,
    radial_resolution=501,
    axial_resolution=1001,
    zoom_fraction=0.15,
    amplitude_floor=1.0e-14,
):
    """
    Plot phase-independent component envelopes along the cylinder height.

    For every z, the displayed value is the maximum component amplitude over
    the radius. The lower panels zoom simultaneously on the bottom and top
    endcaps.
    """

    Gamma = out["Gamma"]
    k = n_axial * np.pi / Gamma

    # Radial grid used to calculate maximum component envelopes.
    theta = np.linspace(0.0, np.pi, radial_resolution)
    s = 0.5 * (1.0 - np.cos(theta))

    z = np.linspace(0.0, Gamma, axial_resolution)
    z_over_h = z / Gamma

    vs_amp, vphi_amp, vz_amp = radial_component_profiles(
        out["solution"],
        s,
        out["indices"],
        m,
        k,
        kind="velocity",
    )

    bs_amp, bphi_amp, bz_amp = radial_component_profiles(
        out["solution"],
        s,
        out["indices"],
        m,
        k,
        kind="magnetic",
    )

    # Maximum radial amplitudes before applying axial parity.
    velocity_radial_max = np.asarray([
        np.max(np.abs(vs_amp)),
        np.max(np.abs(vphi_amp)),
        np.max(np.abs(vz_amp)),
    ])

    magnetic_radial_max = np.asarray([
        np.max(np.abs(bs_amp)),
        np.max(np.abs(bphi_amp)),
        np.max(np.abs(bz_amp)),
    ])

    # Axial parity used by the reconstruction.
    velocity_axial_factors = np.vstack((
        np.abs(np.cos(k * z)),
        np.abs(np.cos(k * z)),
        np.abs(np.sin(k * z)),
    ))

    magnetic_axial_factors = np.vstack((
        np.abs(np.sin(k * z)),
        np.abs(np.sin(k * z)),
        np.abs(np.cos(k * z)),
    ))

    velocity_envelopes = (
        velocity_radial_max[:, None]
        * velocity_axial_factors
    )

    magnetic_envelopes = (
        magnetic_radial_max[:, None]
        * magnetic_axial_factors
    )

    # Independent normalization for velocity and magnetic field.
    velocity_scale = np.max(velocity_envelopes)
    magnetic_scale = np.max(magnetic_envelopes)

    if velocity_scale == 0.0:
        velocity_scale = 1.0

    if magnetic_scale == 0.0:
        magnetic_scale = 1.0

    velocity_envelopes /= velocity_scale
    magnetic_envelopes /= magnetic_scale

    labels_velocity = (
        r"$|v_s|$",
        r"$|v_\varphi|$",
        r"$|v_z|$",
    )

    labels_magnetic = (
        r"$|b_s|$",
        r"$|b_\varphi|$",
        r"$|b_z|$",
    )

    colors = (
        "#2166AC",
        "#D95F02",
        "#1B9E77",
    )

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(12, 7.5),
        constrained_layout=True,
    )

    # ================================================================
    # Complete cylinder height
    # ================================================================

    for profile, label, color in zip(
        velocity_envelopes,
        labels_velocity,
        colors,
    ):
        axes[0, 0].semilogy(
            z_over_h,
            np.maximum(profile, amplitude_floor),
            color=color,
            linewidth=1.8,
            label=label,
        )

    for profile, label, color in zip(
        magnetic_envelopes,
        labels_magnetic,
        colors,
    ):
        axes[0, 1].semilogy(
            z_over_h,
            np.maximum(profile, amplitude_floor),
            color=color,
            linewidth=1.8,
            label=label,
        )

    axes[0, 0].set_title("Velocity over the complete height")
    axes[0, 1].set_title("Magnetic field over the complete height")

    for ax in axes[0]:
        ax.axvline(
            0.0,
            color="black",
            linestyle="--",
            linewidth=1.0,
        )
        ax.axvline(
            1.0,
            color="black",
            linestyle="--",
            linewidth=1.0,
        )
        ax.set(
            xlabel=r"$z/h$",
            xlim=(0.0, 1.0),
            ylim=(amplitude_floor, 1.2),
        )

    axes[0, 0].set_ylabel("normalized maximum amplitude")

    # ================================================================
    # Simultaneous zoom at bottom and top
    # ================================================================

    bottom = z_over_h <= zoom_fraction
    top = z_over_h >= 1.0 - zoom_fraction

    bottom_distance = z_over_h[bottom]
    top_distance = 1.0 - z_over_h[top][::-1]

    for profile, label, color in zip(
        velocity_envelopes,
        labels_velocity,
        colors,
    ):
        axes[1, 0].semilogy(
            bottom_distance,
            np.maximum(
                profile[bottom],
                amplitude_floor,
            ),
            color=color,
            linewidth=1.8,
            label=label + " — bottom",
        )

        axes[1, 0].semilogy(
            top_distance,
            np.maximum(
                profile[top][::-1],
                amplitude_floor,
            ),
            color=color,
            linestyle="--",
            linewidth=1.5,
            label=label + " — top",
        )

    for profile, label, color in zip(
        magnetic_envelopes,
        labels_magnetic,
        colors,
    ):
        axes[1, 1].semilogy(
            bottom_distance,
            np.maximum(
                profile[bottom],
                amplitude_floor,
            ),
            color=color,
            linewidth=1.8,
            label=label + " — bottom",
        )

        axes[1, 1].semilogy(
            top_distance,
            np.maximum(
                profile[top][::-1],
                amplitude_floor,
            ),
            color=color,
            linestyle="--",
            linewidth=1.5,
            label=label + " — top",
        )

    axes[1, 0].set_title("Velocity near the endcaps")
    axes[1, 1].set_title("Magnetic field near the endcaps")

    for ax in axes[1]:
        ax.axvline(
            0.0,
            color="black",
            linestyle=":",
            linewidth=1.0,
        )

        ax.set(
            xlabel=r"distance from endcap$/h$",
            xlim=(0.0, zoom_fraction),
            ylim=(amplitude_floor, 1.2),
        )

    axes[1, 0].set_ylabel("normalized maximum amplitude")

    for ax in axes.ravel():
        ax.grid(
            True,
            which="both",
            alpha=0.22,
        )

        ax.legend(
            frameon=False,
            fontsize=7.5,
        )
    """
    fig.suptitle(
        fr"Axial boundary behaviour for "
        fr"$(m,n)=({m},{n_axial})$",
        fontsize=13,
    )
    """
    # ================================================================
    # Endpoint diagnostics
    # ================================================================

    bottom_velocity = velocity_envelopes[:, 0]
    top_velocity = velocity_envelopes[:, -1]

    bottom_magnetic = magnetic_envelopes[:, 0]
    top_magnetic = magnetic_envelopes[:, -1]

    print("Normalized envelopes at z = 0")
    print(
        f"velocity: |vs|={bottom_velocity[0]:.3e}, "
        f"|vphi|={bottom_velocity[1]:.3e}, "
        f"|vz|={bottom_velocity[2]:.3e}"
    )
    print(
        f"magnetic: |bs|={bottom_magnetic[0]:.3e}, "
        f"|bphi|={bottom_magnetic[1]:.3e}, "
        f"|bz|={bottom_magnetic[2]:.3e}"
    )

    print("\nNormalized envelopes at z = h")
    print(
        f"velocity: |vs|={top_velocity[0]:.3e}, "
        f"|vphi|={top_velocity[1]:.3e}, "
        f"|vz|={top_velocity[2]:.3e}"
    )
    print(
        f"magnetic: |bs|={top_magnetic[0]:.3e}, "
        f"|bphi|={top_magnetic[1]:.3e}, "
        f"|bz|={top_magnetic[2]:.3e}"
    )

    diagnostics = {
        "z_over_h": z_over_h,
        "velocity_envelopes": velocity_envelopes,
        "magnetic_envelopes": magnetic_envelopes,
        "bottom_velocity": bottom_velocity,
        "top_velocity": top_velocity,
        "bottom_magnetic": bottom_magnetic,
        "top_magnetic": top_magnetic,
    }

    return fig, axes, diagnostics

In [ ]:
fig, axes, endcap_diagnostics = plot_endcap_profiles(
    out,
    m=CASE_M,
    n_axial=CASE_N,
)

fig.savefig(OUTPUT_DIR / "Axial_BC.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def plot_meridional_slice_direct(
    out, *, m, n_axial, phi=0.0, tau=0.0, resolution=301,
):
    """Direct spectral evaluation: no griddata or cubic interpolation."""
    Gamma = out["Gamma"]
    k = n_axial*np.pi/Gamma
    common = dict(
        solution=out["solution"], indices=out["indices"],
        m=m, k=k, omega=out["omega"]/out["Omega_rot"],
        height=Gamma, phi=phi, time=tau, resolution=resolution,
    )
    s, z, vs, vphi, vz = meridional_components(**common, kind="velocity")
    _, _, bs, bphi, bz = meridional_components(**common, kind="magnetic")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True,
                             sharey=True, constrained_layout=True)
    panels = ((vphi, vs, vz, "Velocity", r"$v_\varphi$"),
              (bphi, bs, bz, "Magnetic field", r"$b_\varphi$"))
    for ax, (color, radial, axial, title, label) in zip(axes, panels):
        scale = np.max(np.abs(color)) or 1.0
        pcm = ax.pcolormesh(s, z, color, shading="auto", cmap="RdBu_r",
                            vmin=-scale, vmax=scale, rasterized=True)
        ax.streamplot(s, z, radial, axial, color="k", density=1.25,
                      linewidth=0.7, arrowsize=0.8)
        fig.colorbar(pcm, ax=ax, label=label, fraction=0.046, pad=0.04)
        ax.set(xlabel=r"$s$", ylabel=r"$z/a$", title=title,
               xlim=(0, 1), ylim=(0, Gamma))
        ax.set_aspect("equal", adjustable="box")
    return fig, axes

fig_meridional, axes_meridional = plot_meridional_slice_direct(
    out, m=CASE_M, n_axial=CASE_N
)
fig_meridional.savefig(
    OUTPUT_DIR / "Fields_H.png", dpi=300, bbox_inches="tight"
)
plt.show()

## 11. Viscous and Ohmic sensitivity

In [ ]:
# === MULTI-MODE SENSITIVITY TO VISCOUS AND MAGNETIC DIFFUSION ===

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D


# ----------------------------------------------------------------------
# Parameters
# ----------------------------------------------------------------------

DISSIPATION_MODES = (
    (1, 1), (2, 1), (3, 1),
    (1, 2), (2, 2), (3, 2),
)

DISSIPATION_VALUES = np.logspace(-6, -2, 10)
DISSIPATION_N = 48

Gamma = MODEL["h"]/MODEL["a"]

assert FULL_MHD_LE > 0.0
assert FULL_MHD_E > 0.0
assert FULL_MHD_EM > 0.0


# ----------------------------------------------------------------------
# Same colour convention as in the other figures
# ----------------------------------------------------------------------

colors_n1 = plt.cm.Oranges(
    np.linspace(0.45, 0.90, 3)
)

colors_n2 = plt.cm.Blues(
    np.linspace(0.45, 0.90, 3)
)

mode_colors = {
    (1, 1): colors_n1[0],
    (2, 1): colors_n1[1],
    (3, 1): colors_n1[2],
    (1, 2): colors_n2[0],
    (2, 2): colors_n2[1],
    (3, 2): colors_n2[2],
}


# ----------------------------------------------------------------------
# Continuation of one mode
# ----------------------------------------------------------------------

def dissipation_sensitivity(mode, vary):
    """
    Continue one MC mode while varying either E or Em.

    The ideal fast-MC frequency is used only to initialize the first
    numerical solve. Subsequent targets are the previously computed
    dissipative eigenfrequencies.
    """

    if vary not in {"E", "Em"}:
        raise ValueError(
            "vary must be 'E' or 'Em'"
        )

    m, n = mode
    k = n*np.pi/Gamma

    omega_I = greenspan_frequencies(
        m,
        n,
        Gamma,
        branch=1,
        count=1,
    )[1][0]

    target = ideal_magnetocoriolis_frequencies(
        omega_I,
        k,
        FULL_MHD_LE,
    )[0]

    previous_fingerprint = None
    rows = []

    for value in DISSIPATION_VALUES:
        E = (
            value
            if vary == "E"
            else FULL_MHD_E
        )

        Em = (
            value
            if vary == "Em"
            else FULL_MHD_EM
        )

        result = solve_dissipative_mhd_mode(
            resolution=DISSIPATION_N,
            m=m,
            k=k,
            Le=FULL_MHD_LE,
            E=E,
            Em=Em,
            target=target,
            previous_fingerprint=previous_fingerprint,
            mechanical_boundary="no_slip",
            frequency_window=1.0,
        )

        rows.append(result)

        # True numerical continuation.
        target = result.frequency
        previous_fingerprint = result.fingerprint

    return rows


# ----------------------------------------------------------------------
# Compute all modes
# ----------------------------------------------------------------------

dissipation_results = {}

for mode in DISSIPATION_MODES:
    print(f"Computing mode {mode} ...")

    dissipation_results[mode] = {
        "E": dissipation_sensitivity(
            mode,
            "E",
        ),
        "Em": dissipation_sensitivity(
            mode,
            "Em",
        ),
    }


# ----------------------------------------------------------------------
# Numerical diagnostics
# ----------------------------------------------------------------------

print()
print("CONTINUATION DIAGNOSTICS")

for mode, mode_results in dissipation_results.items():
    print()
    print(f"Mode {mode}")

    for vary, rows in mode_results.items():
        overlaps = np.asarray([
            row.overlap_with_previous
            for row in rows[1:]
            if row.overlap_with_previous
            is not None
        ])

        minimum_overlap = (
            np.min(overlaps)
            if overlaps.size
            else np.nan
        )

        maximum_wall_residual = max(
            row.boundary_residual
            for row in rows
        )

        maximum_backward_residual = max(
            row.backward_residual
            for row in rows
        )

        print(
            f"  vary {vary:2s}: "
            f"minimum overlap = "
            f"{minimum_overlap:.6f}, "
            f"maximum wall residual = "
            f"{maximum_wall_residual:.3e}, "
            f"maximum backward residual = "
            f"{maximum_backward_residual:.3e}"
        )

        if minimum_overlap < 0.95:
            print(
                "    WARNING: possible branch "
                "switch or insufficient sampling."
            )


In [ ]:
# ----------------------------------------------------------------------
# Figure: damping on the left, frequencies separated by axial index
# ----------------------------------------------------------------------

fig = plt.figure(
    figsize=(12, 5),
    constrained_layout=True,
)

grid = fig.add_gridspec(
    2,
    2,
    width_ratios=(1.12, 1.0),
)

ax_damping = fig.add_subplot(
    grid[:, 0]
)

ax_frequency_n2 = fig.add_subplot(
    grid[0, 1]
)

ax_frequency_n1 = fig.add_subplot(
    grid[1, 1],
    sharex=ax_frequency_n2,
)

sweep_styles = {
    "E": {
        "linestyle": "-",
        "marker": "o",
    },
    "Em": {
        "linestyle": "--",
        "marker": "s",
    },
}


# ----------------------------------------------------------------------
# Plot all modes
# ----------------------------------------------------------------------

for mode in DISSIPATION_MODES:
    m, n = mode
    color = mode_colors[mode]

    frequency_axis = (
        ax_frequency_n2
        if n == 2
        else ax_frequency_n1
    )

    for vary in ("E", "Em"):
        rows = dissipation_results[mode][vary]

        omega = np.asarray([
            row.frequency
            for row in rows
        ])

        damping = -omega.imag
        style = sweep_styles[vary]

        if np.any(damping <= 0.0):
            print(
                f"Warning: non-positive damping "
                f"for mode {mode}, varying {vary}."
            )

        # Damping: all modes in the same panel.
        ax_damping.loglog(
            DISSIPATION_VALUES,
            np.maximum(damping, 1.0e-16),
            color=color,
            linestyle=style["linestyle"],
            marker=style["marker"],
            linewidth=1.5,
            markersize=3.8,
        )

        # Frequency: separate panels for n=1 and n=2.
        frequency_axis.semilogx(
            DISSIPATION_VALUES,
            omega.real,
            color=color,
            linestyle=style["linestyle"],
            marker=style["marker"],
            linewidth=1.5,
            markersize=3.8,
        )


# ----------------------------------------------------------------------
# Axis labels and formatting
# ----------------------------------------------------------------------

ax_damping.set(
    xlabel="varied diffusivity",
    ylabel=r"$-\Im(\omega)$",
    title="Damping sensitivity",
)

ax_frequency_n2.set(
    ylabel=r"$\Re(\omega)$",
    title=r"Frequency sensitivity: $n=2$",
)

ax_frequency_n1.set(
    xlabel="varied diffusivity",
    ylabel=r"$\Re(\omega)$",
    title=r"Frequency sensitivity: $n=1$",
)

# Hide duplicated x labels on the upper-right panel.
ax_frequency_n2.tick_params(
    axis="x",
    labelbottom=False,
)

for ax in (
    ax_damping,
    ax_frequency_n2,
    ax_frequency_n1,
):
    ax.grid(
        True,
        which="both",
        alpha=0.25,
    )

    ax.margins(y=0.10)


# ----------------------------------------------------------------------
# Legend handles
# ----------------------------------------------------------------------

mode_handles_all = [
    Line2D(
        [0],
        [0],
        color=mode_colors[mode],
        linewidth=2,
        label=fr"$(m,n)={mode}$",
    )
    for mode in DISSIPATION_MODES
]

mode_handles_n2 = [
    Line2D(
        [0],
        [0],
        color=mode_colors[mode],
        linewidth=2,
        label=fr"$m={mode[0]}$",
    )
    for mode in DISSIPATION_MODES
    if mode[1] == 2
]

mode_handles_n1 = [
    Line2D(
        [0],
        [0],
        color=mode_colors[mode],
        linewidth=2,
        label=fr"$m={mode[0]}$",
    )
    for mode in DISSIPATION_MODES
    if mode[1] == 1
]

sweep_handles = [
    Line2D(
        [0],
        [0],
        color="0.20",
        linestyle="-",
        marker="o",
        linewidth=1.5,
        markersize=4,
        label=fr"vary $E$ "
              fr"($E_m={FULL_MHD_EM:.0e}$)",
    ),
    Line2D(
        [0],
        [0],
        color="0.20",
        linestyle="--",
        marker="s",
        linewidth=1.5,
        markersize=4,
        label=fr"vary $E_m$ "
              fr"($E={FULL_MHD_E:.0e}$)",
    ),
]


# ----------------------------------------------------------------------
# Legends
# ----------------------------------------------------------------------

mode_legend = ax_damping.legend(
    handles=mode_handles_all,
    frameon=False,
    ncol=2,
    fontsize=8.5,
    loc="upper left",
)

ax_damping.add_artist(mode_legend)

ax_damping.legend(
    handles=sweep_handles,
    frameon=False,
    fontsize=8.5,
    loc="lower right",
)

ax_frequency_n2.legend(
    handles=mode_handles_n2,
    frameon=False,
    ncol=3,
    fontsize=8.5,
    loc="best",
)

ax_frequency_n1.legend(
    handles=mode_handles_n1,
    frameon=False,
    ncol=3,
    fontsize=8.5,
    loc="best",
)

fig.savefig(OUTPUT_DIR / "E_Ek_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. TRANSITION INERTIAL -> MAGNETO–CORIOLIS -> ALFVÉN

In [ ]:
# === TRANSITION INERTIAL -> MAGNETO–CORIOLIS -> ALFVÉN ===

MC_LIMIT_MODES = (
    (1, 1), (2, 1), (3, 1),
    (1, 2), (2, 2), (3, 2),
)

MC_LIMIT_LE = np.logspace(-2, 1, 18)

IDEAL_RESOLUTION = 32
DISSIPATIVE_RESOLUTION = 48

MC_LIMIT_E = PAPER_CASE["E"]
MC_LIMIT_EM = PAPER_CASE["Em"]

Gamma = MODEL["h"] / MODEL["a"]


# Orange for n=1, blue for n=2.
colors_n1 = plt.cm.Oranges(np.linspace(0.45, 0.90, 3))
colors_n2 = plt.cm.Blues(np.linspace(0.45, 0.90, 3))

mode_colors = {
    (1, 1): colors_n1[0],
    (2, 1): colors_n1[1],
    (3, 1): colors_n1[2],
    (1, 2): colors_n2[0],
    (2, 2): colors_n2[1],
    (3, 2): colors_n2[2],
}


def first_inertial_frequency(m, n):
    """First positive Kelvin–Greenspan frequency."""
    _, frequencies = greenspan_frequencies(
        m=m,
        n=n,
        Gamma=Gamma,
        branch=1,
        count=1,
    )
    return frequencies[0]


def numerical_ideal_fast_frequency(m, n, Le):
    """Numerical fast ideal-MC eigenfrequency."""
    rows = ideal_mc_benchmark(
        m=m,
        n=n,
        Gamma=Gamma,
        Le=Le,
        resolution=IDEAL_RESOLUTION,
        radial_count=1,
        inertial_branches=(1,),
    )

    row = next(
        row for row in rows
        if row.radial_index == 1
        and row.mc_branch == "fast"
    )

    return row.numerical_frequency.real


def dissipative_fast_continuation(m, n):
    """Continue the fast MC mode while increasing Le."""
    k = n * np.pi / Gamma
    omega_I = first_inertial_frequency(m, n)

    frequencies = []
    previous_fingerprint = None

    for Le in MC_LIMIT_LE:
        ideal_fast = ideal_magnetocoriolis_frequencies(
            omega_I,
            k,
            Le,
        )[0]

        result = solve_dissipative_mhd_mode(
            resolution=DISSIPATIVE_RESOLUTION,
            m=m,
            k=k,
            Le=Le,
            E=MC_LIMIT_E,
            Em=MC_LIMIT_EM,
            target=ideal_fast,
            previous_fingerprint=previous_fingerprint,
        )

        frequencies.append(result.frequency)
        previous_fingerprint = result.fingerprint

    return np.asarray(frequencies)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)


for mode in MC_LIMIT_MODES:
    m, n = mode
    color = mode_colors[mode]

    k = n * np.pi / Gamma
    omega_I = first_inertial_frequency(m, n)

    # Exact ideal-MC fast branch.
    omega_ideal_exact = np.asarray([
        ideal_magnetocoriolis_frequencies(
            omega_I,
            k,
            Le,
        )[0]
        for Le in MC_LIMIT_LE
    ])

    # Numerical ideal-MHD eigenvalues.
    omega_ideal_numerical = np.asarray([
        numerical_ideal_fast_frequency(m, n, Le)
        for Le in MC_LIMIT_LE
    ])

    # Numerical viscous–diffusive continuation.
    omega_dissipative = dissipative_fast_continuation(m, n)

    # ------------------------------------------------------------------
    # Ideal MHD
    # ------------------------------------------------------------------

    axes[0].loglog(
        MC_LIMIT_LE,
        np.abs(omega_ideal_exact),
        color=color,
        lw=2,
        label=fr"$(m,n)={mode}$",
    )

    axes[0].loglog(
        MC_LIMIT_LE,
        np.abs(omega_ideal_numerical),
        "k:",
        lw=0.9,
        alpha=0.8,
    )

    # Kelvin–Greenspan inertial limit.
    axes[0].axhline(
        abs(omega_I),
        color=color,
        ls="--",
        lw=1,
        alpha=0.75,
    )

    # ------------------------------------------------------------------
    # Viscous–diffusive MHD
    # ------------------------------------------------------------------

    axes[1].loglog(
        MC_LIMIT_LE,
        np.abs(omega_dissipative.real),
        color=color,
        lw=2,
        label=fr"$(m,n)={mode}$",
    )

    # Same ideal inertial reference.
    axes[1].axhline(
        abs(omega_I),
        color=color,
        ls="--",
        lw=1,
        alpha=0.75,
    )


# Alfvén asymptotes depend only on n through k = n*pi/Gamma.
for n, offset in ((1, 1.0), (2, 1.0)):
    k = n * np.pi / Gamma

    for ax in axes:
        ax.loglog(
            MC_LIMIT_LE,
            MC_LIMIT_LE * k,
            color="0.65",
            ls="--",
            lw=1.3,
        )


axes[0].set_title(r"Ideal MHD ($E=E_m=0$)")

axes[1].set_title(
    fr"Viscous–diffusive MHD "
    fr"($E={MC_LIMIT_E:.0e}$, $E_m={MC_LIMIT_EM:.0e}$)"
)

axes[0].set_ylabel(r"$|\Re(\omega)|/\Omega$")

for ax in axes:
    ax.set_xlabel(r"$Le$")
    ax.grid(True, which="both", alpha=0.3)

axes[0].legend(
    frameon=False,
    ncol=2,
    fontsize=8,
)

fig.savefig(OUTPUT_DIR / "omega_Le.png", dpi=300, bbox_inches="tight")
plt.show()